In [2]:
import sys
import os
import glob

# 1. Khai báo đường dẫn SPARK_HOME
os.environ['SPARK_HOME'] = '/home/quang/spark-4.1.1-bin-hadoop3'

# 2. Thêm thư mục mã nguồn python của Spark vào hệ thống
sys.path.insert(0, '/home/quang/spark-4.1.1-bin-hadoop3/python')

# 3. TỰ ĐỘNG TÌM VÀ THÊM THƯ VIỆN PY4J
try:
    py4j_zip = glob.glob(os.path.join(os.environ['SPARK_HOME'], 'python/lib', 'py4j-*.zip'))[0]
    sys.path.insert(0, py4j_zip)
except IndexError:
    print("Cảnh báo: Không tìm thấy file py4j.zip trong thư mục spark/python/lib")

# 4. Bây giờ tiến hành khởi tạo Spark Session trực tiếp
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab3_BigData") \
    .master("local[*]") \
    .getOrCreate()

# 5. Kiểm tra thành quả
print("Kết nối thành công! Phiên bản Spark của bạn là:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 17:31:29 WARN Utils: Your hostname, quang-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/05/21 17:31:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 17:31:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Kết nối thành công! Phiên bản Spark của bạn là: 4.1.1


In [6]:
sc = spark.sparkContext
movies = sc.textFile("file:///media/sf_DS200_BigData/Lab3/movies.txt")
ratings1 = sc.textFile("file:///media/sf_DS200_BigData/Lab3/ratings_1.txt")
ratings2 = sc.textFile("file:///media/sf_DS200_BigData/Lab3/ratings_2.txt")
users = sc.textFile("file:///media/sf_DS200_BigData/Lab3/users.txt")
occupations = sc.textFile("file:///media/sf_DS200_BigData/Lab3/occupation.txt")

## Bai 1

In [23]:
movie_map = movies.map(lambda line: line.split(","))\
    .map(lambda fields: (fields[0], fields[1])) \
    .collectAsMap()
ratings = ratings1.union(ratings2)
movie_ratings = ratings.map(lambda line: line.split(",")) \
    .map(lambda fields: (fields[1], (float(fields[2]), 1)))

movie_stats = movie_ratings.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

movie_avg = movie_stats.mapValues(lambda x: (x[0] / x[1], x[1]))

popular5_movies = movie_avg.filter(lambda kv: kv[1][1] >= 5)
popular50_movies = movie_avg.filter(lambda kv: kv[1][1] >= 50)

best_movie = popular5_movies.max(key=lambda kv: kv[1][0])
print(popular50_movies.take(1))
print(best_movie)
print(movie_map.get(best_movie[0], "Unknown"), best_movie[1])

[]
('1015', (4.357142857142857, 7))
Sunset Boulevard (1950) (4.357142857142857, 7)


## Bai 2

In [28]:
movie_genres = movies.map(
    lambda line: line.split(",")
).flatMap(
    lambda fields: [(fields[0], genre) for genre in fields[2].split("|")]
).groupByKey().mapValues(list).collectAsMap()

genre_ratings = ratings.map(
    lambda line: line.split(",") 
).flatMap(
    lambda fields: [
        (genre, (float(fields[2]), 1))
        for genre in movie_genres.get(fields[1], [])
    ]
)
genre_stats = genre_ratings.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)
genre_avg = genre_stats.mapValues(lambda x: x[0] / x[1])
for genre, avg_rating in genre_avg.collect():
    print(f"Thể loại: {genre} - Điểm trung bình: {avg_rating:.2f}")

[Stage 54:=============================>                            (2 + 2) / 4]

Thể loại: Family - Điểm trung bình: 3.67
Thể loại: Sci-Fi - Điểm trung bình: 3.73
Thể loại: Drama - Điểm trung bình: 3.76
Thể loại: Thriller - Điểm trung bình: 3.70
Thể loại: Fantasy - Điểm trung bình: 3.86
Thể loại: Film-Noir - Điểm trung bình: 4.36
Thể loại: Adventure - Điểm trung bình: 3.63
Thể loại: Biography - Điểm trung bình: 3.56
Thể loại: Horror - Điểm trung bình: 4.00
Thể loại: Action - Điểm trung bình: 3.71
Thể loại: Crime - Điểm trung bình: 3.81
Thể loại: Mystery - Điểm trung bình: 4.00


In [ ]:
## Bai 3

In [27]:
user_gender = users.map(lambda line: line.split(",")) \
    .map(lambda fields: (fields[0], fields[1])) \
    .collectAsMap()
movie_gender_ratings = ratings.map(lambda line: line.split(",")) \
    .map(lambda fields: ((fields[1], user_gender.get(fields[0], "Unknown")), (float(fields[2]), 1)))
movie_gender_stats = movie_gender_ratings.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
movie_gender_avg = movie_gender_stats.mapValues(lambda x: x[0] / x[1])
print("Điểm trung bình của mỗi phim theo giới tính:")
for key, avg_rating in movie_gender_avg.collect():
    print(f"Phim ID: {key[0]}, Giới tính: {key[1]}, Điểm trung bình: {avg_rating:.2f}")

Điểm trung bình của mỗi phim theo giới tính:
Phim ID: 1012, Giới tính: F, Điểm trung bình: 4.00
Phim ID: 1025, Giới tính: F, Điểm trung bình: 4.14
Phim ID: 1043, Giới tính: M, Điểm trung bình: 3.92
Phim ID: 1013, Giới tính: F, Điểm trung bình: 3.94
Phim ID: 1043, Giới tính: F, Điểm trung bình: 3.83
Phim ID: 1013, Giới tính: M, Điểm trung bình: 4.06
Phim ID: 1025, Giới tính: M, Điểm trung bình: 3.93
Phim ID: 1039, Giới tính: F, Điểm trung bình: 3.90
Phim ID: 1039, Giới tính: M, Điểm trung bình: 3.75
Phim ID: 1030, Giới tính: M, Điểm trung bình: 3.33
Phim ID: 1047, Giới tính: F, Điểm trung bình: 3.67
Phim ID: 1050, Giới tính: F, Điểm trung bình: 3.32
Phim ID: 1010, Giới tính: M, Điểm trung bình: 3.55
Phim ID: 1028, Giới tính: M, Điểm trung bình: 3.50
Phim ID: 1030, Giới tính: F, Điểm trung bình: 3.00
Phim ID: 1047, Giới tính: M, Điểm trung bình: 4.00
Phim ID: 1010, Giới tính: F, Điểm trung bình: 3.31
Phim ID: 1028, Giới tính: F, Điểm trung bình: 3.50
Phim ID: 1050, Giới tính: M, Điểm tru

## Bai 4

In [31]:
def age_group(age):
    age = int(age)
    if age < 18:
        return "<18"
    if age < 25:
        return "18-24"
    if age < 35:
        return "25-34"
    if age < 45:
        return "35-44"
    if age < 55:
        return "45-54"
    return "55+"

user_age_group = users.map(lambda line: line.split(","))\
    .map(lambda fields: (fields[0], age_group(fields[2]))) \
    .collectAsMap()

movie_age_ratings = ratings.map(lambda line: line.split(",")) \
    .map(lambda fields: ((fields[1], user_age_group.get(fields[0], "Unknown")), (float(fields[2]), 1)))

movie_age_stats = movie_age_ratings.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
movie_age_avg = movie_age_stats.mapValues(lambda x: x[0] / x[1])
for key, avg_rating in movie_age_avg.collect():
    print(f"Phim ID: {key[0]}, Nhóm tuổi: {key[1]}, Điểm trung bình: {avg_rating:.2f}")

Phim ID: 1020, Nhóm tuổi: 35-44, Điểm trung bình: 3.81
Phim ID: 1040, Nhóm tuổi: 55+, Điểm trung bình: 3.00
Phim ID: 1043, Nhóm tuổi: 25-34, Điểm trung bình: 3.83
Phim ID: 1040, Nhóm tuổi: 35-44, Điểm trung bình: 3.75
Phim ID: 1025, Nhóm tuổi: 25-34, Điểm trung bình: 4.10
Phim ID: 1013, Nhóm tuổi: 25-34, Điểm trung bình: 3.71
Phim ID: 1050, Nhóm tuổi: 18-24, Điểm trung bình: 3.00
Phim ID: 1037, Nhóm tuổi: 35-44, Điểm trung bình: 3.69
Phim ID: 1015, Nhóm tuổi: 35-44, Điểm trung bình: 4.50
Phim ID: 1043, Nhóm tuổi: 45-54, Điểm trung bình: 4.38
Phim ID: 1039, Nhóm tuổi: 45-54, Điểm trung bình: 4.00
Phim ID: 1020, Nhóm tuổi: 55+, Điểm trung bình: 3.00
Phim ID: 1039, Nhóm tuổi: 25-34, Điểm trung bình: 3.83
Phim ID: 1025, Nhóm tuổi: 45-54, Điểm trung bình: 4.17
Phim ID: 1047, Nhóm tuổi: 35-44, Điểm trung bình: 3.50
Phim ID: 1012, Nhóm tuổi: 25-34, Điểm trung bình: 4.50
Phim ID: 1010, Nhóm tuổi: 35-44, Điểm trung bình: 3.28
Phim ID: 1030, Nhóm tuổi: 35-44, Điểm trung bình: 3.00
Phim ID: 1015,

## Bai 5

In [32]:
occupation_map = occupations.map(lambda line: line.split(",")) \
    .map(lambda fields: (fields[0], fields[1])) \
    .collectAsMap()

user_occupation = users.map(lambda line: line.split(",")) \
    .map(lambda fields: (fields[0], occupation_map.get(fields[3], "Unknown"))) \
    .collectAsMap()

occupation_ratings = ratings.map(lambda line: line.split(",")) \
    .map(lambda fields: (user_occupation.get(fields[0], "Unknown"), (float(fields[2]), 1)))

occupation_stats = occupation_ratings.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
occupation_avg = occupation_stats.mapValues(lambda x: (x[0] / x[1], x[1]))
for occ, (avg_rating, total_count) in occupation_avg.collect():
    print(f"Nghề: {occ} - Điểm TB: {avg_rating:.2f} - Lượt đánh giá: {total_count}")

[Stage 67:=============================>                            (2 + 2) / 4]

Nghề: Nurse - Điểm TB: 3.86 - Lượt đánh giá: 11
Nghề: Manager - Điểm TB: 3.47 - Lượt đánh giá: 16
Nghề: Artist - Điểm TB: 3.73 - Lượt đánh giá: 11
Nghề: Designer - Điểm TB: 4.00 - Lượt đánh giá: 13
Nghề: Programmer - Điểm TB: 4.25 - Lượt đánh giá: 10
Nghề: Teacher - Điểm TB: 3.70 - Lượt đánh giá: 5
Nghề: Consultant - Điểm TB: 3.86 - Lượt đánh giá: 14
Nghề: Doctor - Điểm TB: 3.69 - Lượt đánh giá: 21
Nghề: Student - Điểm TB: 4.00 - Lượt đánh giá: 8
Nghề: Salesperson - Điểm TB: 3.65 - Lượt đánh giá: 17
Nghề: Engineer - Điểm TB: 3.56 - Lượt đánh giá: 18
Nghề: Journalist - Điểm TB: 3.85 - Lượt đánh giá: 17
Nghề: Lawyer - Điểm TB: 3.65 - Lượt đánh giá: 17
Nghề: Accountant - Điểm TB: 3.58 - Lượt đánh giá: 6


## Bai 6

In [33]:
import datetime

def get_year(ts):
    return datetime.datetime.fromtimestamp(int(ts)).year

rating_years = ratings.map(lambda line: line.split(","))\
    .map(lambda fields: (get_year(fields[3]), (float(fields[2]), 1)))

year_stats = rating_years.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
year_avg = year_stats.mapValues(lambda x: (x[0] / x[1], x[1]))
for year, (avg_rating, total_count) in sorted(year_avg.collect()):
    print(f"Năm: {year} - Điểm TB: {avg_rating:.2f} - Tổng lượt: {total_count}")

[Stage 69:===========================================>              (3 + 1) / 4]

Năm: 2020 - Điểm TB: 3.75 - Tổng lượt: 184
